# Diários Oficiais do Rio — Downloader

> Notebook standalone para baixar PDFs dos diários oficiais do Município do Rio de Janeiro de forma automática.

## Por quê?

Quem acompanha política municipal carioca precisa cruzar dois diários oficiais quase todo dia:

- **DOP** — *Diário Oficial da Prefeitura* (`doweb.rio.rj.gov.br`) → atos do Executivo: decretos, gastos, contratos, nomeações.
- **DCM** — *Diário da Câmara Municipal* (`dcmdigital.camara.rj.gov.br`) → atos do Legislativo: projetos de lei, pareceres, discursos, votações.

Os dois portais oferecem download manual no navegador, mas não têm API pública. Esse notebook automatiza a etapa — três modos de uso, uma fonte ou as duas, configurável em uma célula.

## Modos suportados

| Modo | O que significa | DCM | DOP |
|---|---|---|---|
| `hoje` | última edição publicada agora no portal | ✅ | ✅ |
| `edicao` **por número** | "DCM nº 108 de 2025" | ✅ via formulário (Selenium) | ⚠ só edições recentes (ver Seção 3) |
| `edicao` **por data** | "DCM/DOP publicado em 2025-03-31" | ✅ heurística + verificação no PDF | ⚠ só edições recentes |
| `range` **por número** | "DCM 156 a 167 de 2025" | ✅ | ⚠ só edições recentes |
| `range` **por data** | "DCM/DOP de 24 a 30 de março de 2025" | ✅ pula sábado/domingo | ⚠ só edições recentes |

> O ⚠ no DOP é uma limitação do portal, não do código: a home só expõe a "última data" e seus suplementos. Pra arquivo histórico do DOP é preciso descobrir o ID manualmente no devtools (instruções na Seção 3).

## Como funciona internamente

- **DOP** → 100% HTTP. A home expõe a edição corrente numa variável JS `DADOS_ULTIMA_DATA`; um regex extrai o `id` e um `GET /portal/edicoes/download/{id}` baixa o PDF.
- **DCM** → Selenium headless. A página é uma SPA: pra ler a edição atual ou usar o formulário de busca precisa de browser real. O site não tem busca por data — `baixar_dcm_por_data` palpita o número (contando dias úteis a partir da edição atual) e confirma lendo a primeira página do PDF. Tem retry automático porque o CDN do DCM cai em `SSL: UNEXPECTED_EOF_WHILE_READING` de vez em quando.

## Como usar — 5 minutos

1. **Primeira vez?** Na Seção 2 (Instalação), descomente as linhas `# !pip install …` e `# !apt-get install …` e rode essa célula. Depois pode comentar de novo se quiser.
2. **Edite só a Seção 1** (`MODO`, `FONTE`, parâmetros). Os defaults baixam DCM + DOP de hoje em `./downloads/` — funciona sem você mexer em nada.
3. **Run All** (`Cell → Run All` no Jupyter, ou shift+enter por célula até o fim).
4. **Confira na Seção 6** a tabela com os PDFs baixados (nome, MB, páginas).

Se algo falhar, vá direto pra **Seção 7 (Troubleshooting)** — os erros mais comuns têm receita ali.

---
*Validado em CI (GitHub Actions, Ubuntu + Chrome stable). Parte do projeto [clipping-project](https://github.com/OttoBoop/clipping-project) — pipeline de clipping político carioca.*


## 1. Configuração

A única célula que você precisa editar. Lê de cima pra baixo:

- **MODO** — o que baixar: a edição de hoje, uma específica, ou um intervalo.
- **FONTE** — de qual diário: só DCM, só DOP, ou os dois.
- **EDICAO_\***, **RANGE_\*** — parâmetros usados conforme o modo.
- **DESTINO** — pasta onde os PDFs caem.

### Exemplos rápidos

```python
# Tudo de hoje (mais comum)
MODO, FONTE = "hoje", "ambos"

# DCM edição 108 de 2025 (rota confiável — busca direta no portal)
MODO, FONTE = "edicao", "dcm"
EDICAO_NUMERO, EDICAO_ANO = 108, 2025
EDICAO_DATA = None

# DCM publicado em 31/03/2025 (heurística + verificação na 1ª página)
MODO, FONTE = "edicao", "dcm"
EDICAO_DATA = "2025-03-31"

# DOP do dia 31/03/2025 (precisa estar disponível no portal, veja limitação na Seção 3)
MODO, FONTE = "edicao", "dop"
EDICAO_DATA = "2025-03-31"

# DCM edições 156 a 167 de 2025
MODO, FONTE = "range", "dcm"
RANGE_TIPO, RANGE_INICIO, RANGE_FIM, EDICAO_ANO = "edicao", 156, 167, 2025

# DCM de uma semana (pula sábado e domingo automaticamente)
MODO, FONTE = "range", "dcm"
RANGE_TIPO, RANGE_INICIO, RANGE_FIM = "data", "2025-03-24", "2025-03-30"

# DOP de uma semana
MODO, FONTE = "range", "dop"
RANGE_TIPO, RANGE_INICIO, RANGE_FIM = "data", "2025-03-24", "2025-03-30"
```

### O que esperar

| Cenário | Tempo típico | Tamanho típico |
|---|---|---|
| `MODO="hoje"`, `FONTE="dop"` | ~5–15 s | 1 PDF, 5–15 MB |
| `MODO="hoje"`, `FONTE="dcm"` | ~30–60 s (sobe Chrome) | 1 PDF unificado, 5–30 MB |
| `MODO="hoje"`, `FONTE="ambos"` | ~40–80 s | 2 PDFs |
| `MODO="edicao"` DCM por número | ~30–60 s | 1 PDF, 5–30 MB |
| `MODO="edicao"` DCM por data | ~40–90 s (1–5 tentativas) | 1 PDF (sufixo `_palpite` se não conseguiu confirmar) |
| `MODO="range"` 5 dias úteis DCM | ~3–6 min | 5 PDFs (algumas falhas individuais são toleradas) |

Tempos medidos em CI (GitHub Actions, Ubuntu + Chrome stable). Local com Chrome instalado costuma ser mais rápido.


In [ ]:
# ==========================================================
#  O QUE BAIXAR
# ==========================================================
MODO   = "hoje"      # "hoje" | "edicao" | "range"
FONTE  = "ambos"     # "dcm"  | "dop"    | "ambos"

# --- usado se MODO == "edicao" ---
# Você escolhe por NÚMERO (EDICAO_NUMERO + EDICAO_ANO) ou por DATA (EDICAO_DATA).
# Se ambos estiverem setados, a DATA tem precedência (tanto em DCM quanto em DOP).
EDICAO_NUMERO = 108        # número da edição (rota direta no DCM)
EDICAO_ANO    = 2025
EDICAO_DATA   = None       # "YYYY-MM-DD" ou None

# --- usado se MODO == "range" ---
# RANGE_TIPO controla como interpretar INICIO/FIM em AMBOS os DOs.
RANGE_TIPO   = "edicao"    # "edicao" (DCM e DOP) | "data" (DCM e DOP)
RANGE_INICIO = 156         # int se tipo='edicao' | "YYYY-MM-DD" se tipo='data'
RANGE_FIM    = 158

# ==========================================================
#  ONDE SALVAR
# ==========================================================
DESTINO = "./downloads"    # ex.: "/content/drive/MyDrive/DOs" no Colab


## 2. Instalação

**Quando descomentar:**

| Ambiente | `pip install` | `apt install chromium` |
|---|---|---|
| Google Colab (primeira vez) | sim | sim, mas use `chromium-chromedriver` ou rode em ambiente com Chrome real |
| Máquina local com Chrome instalado | só os pacotes que faltarem | não precisa |
| GitHub Actions (CI) | sim | use a action `browser-actions/setup-chrome@v1` em vez do apt |

**Importante:** o Selenium 4.6+ usa `selenium-manager` internamente, que baixa o ChromeDriver compatível sozinho. Você só precisa garantir que o **binário do Chrome** esteja instalado e acessível.


In [ ]:
# !pip install --quiet requests beautifulsoup4 PyPDF2 selenium pandas
# !apt-get install -y chromium-chromedriver   # necessário apenas pro DCM


### Imports e setup do destino

Tudo é stdlib + 3 libs externas (`requests`, `PyPDF2`, `selenium` — `pandas` só na última célula pra mostrar resumo). O `urllib3.disable_warnings` silencia avisos do CDN do DCM que ocasionalmente cai em fallback `verify=False`.

In [ ]:
import os, re, json, time, shutil, datetime
from urllib.parse import urljoin

try:
    import requests
    from PyPDF2 import PdfReader, PdfMerger
    import urllib3
except ImportError as e:
    raise ImportError(
        f"Falta o pacote `{e.name}`. Volte na célula `## 2. Instalação`, "
        "descomente as linhas de `pip install` e rode antes de seguir. "
        "Pacotes necessários: requests, PyPDF2, selenium, pandas."
    ) from e

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

os.makedirs(DESTINO, exist_ok=True)
print(f"Destino: {os.path.abspath(DESTINO)}")


## 3. DOP — Diário Oficial da Prefeitura

Fonte: <https://doweb.rio.rj.gov.br>

### Estratégia

100% HTTP, sem Selenium. A home do portal entrega o HTML com uma variável JavaScript embutida:

```js
let DADOS_ULTIMA_DATA = {
  "itens": [
    { "id": 14807, "suplemento": "", ... },    // ← caderno principal
    { "id": 14808, "suplemento": "I", ... },   // ← suplementos do mesmo dia
    ...
  ]
};
```

Pegamos via regex, escolhemos o item com `suplemento == ""` (o caderno principal), e baixamos via `GET /portal/edicoes/download/{id}`.

### Funções públicas

| Função | Modo |
|---|---|
| `baixar_dop_hoje()` | `MODO="hoje"` |
| `baixar_dop_por_data("YYYY-MM-DD")` | `MODO="edicao"` com `EDICAO_DATA` |
| `baixar_dop_por_edicao(numero)` | `MODO="edicao"` com `EDICAO_NUMERO` |
| `baixar_dop_range(inicio, fim, tipo)` | `MODO="range"` |

### Limitação conhecida — arquivo histórico

`baixar_dop_por_data` e `baixar_dop_por_edicao` só funcionam para itens que ainda estejam expostos em `DADOS_ULTIMA_DATA` da home. **Datas/edições antigas não são acessíveis por essa rota.** O portal não documenta endpoint público de arquivo histórico.

Pra contornar: abra o portal no navegador, navegue até a edição desejada, inspecione o link `/portal/edicoes/download/{id}` no devtools e use o ID direto chamando `_dop_baixar_pdf(session, id, nome, destino)`.


In [ ]:
DOP_BASE = "https://doweb.rio.rj.gov.br"
DOP_HEADERS = {
    # UA real é necessário — o portal devolve 403 pra UAs vazios/curl
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

def _dop_pegar_itens_home(session):
    """Lê DADOS_ULTIMA_DATA do HTML da home e retorna a lista de itens (cadernos)."""
    r = session.get(DOP_BASE, headers=DOP_HEADERS, timeout=30)
    r.raise_for_status()
    m = re.search(r"let DADOS_ULTIMA_DATA = (\{.*?\});", r.text, re.DOTALL)
    if not m:
        raise RuntimeError(
            f"Variável DADOS_ULTIMA_DATA não encontrada na home do DOP ({DOP_BASE}). "
            "O portal pode ter mudado a estrutura. Abra a home no devtools, "
            "procure por 'DADOS_ULTIMA_DATA' no <script> e ajuste o regex em "
            "_dop_pegar_itens_home."
        )
    return json.loads(m.group(1)).get("itens", [])

def _dop_baixar_pdf(session, item_id, nome_arquivo, destino):
    """Baixa um PDF pelo ID interno do portal. Valida Content-Type pra evitar
    salvar HTML de erro como .pdf."""
    url = f"{DOP_BASE}/portal/edicoes/download/{item_id}"
    print(f"[DOP] baixando id={item_id} <- {url}")
    r = session.get(url, headers=DOP_HEADERS, timeout=120)
    r.raise_for_status()
    ct = r.headers.get("Content-Type", "")
    if "application/pdf" not in ct.lower():
        raise RuntimeError(
            f"DOP id={item_id} não retornou PDF (Content-Type={ct!r}). "
            "Provavelmente HTML de erro do portal. "
            f"Abra {url} no navegador pra ver o que ele está respondendo."
        )
    caminho = os.path.join(destino, nome_arquivo)
    with open(caminho, "wb") as f:
        f.write(r.content)
    print(f"[DOP] OK {caminho} ({len(r.content)/1024/1024:.2f} MB)")
    return caminho

def baixar_dop_hoje(destino=None):
    """Baixa a edição mais recente publicada do DOP (caderno principal, sem suplemento)."""
    destino = destino or DESTINO
    with requests.Session() as s:
        itens = _dop_pegar_itens_home(s)
        # caderno principal = suplemento vazio (suplementos têm "I", "II", etc)
        item = next((i for i in itens if i.get("suplemento") == ""), None)
        if not item:
            raise RuntimeError(
                "Nenhum caderno principal (suplemento vazio) encontrado na home do DOP. "
                "Pode ser que a edição de hoje ainda não foi publicada — "
                f"confira em {DOP_BASE}."
            )
        data_str = datetime.date.today().strftime("%Y_%m_%d")
        return _dop_baixar_pdf(s, item["id"], f"DOP_{data_str}.pdf", destino)

def baixar_dop_por_data(data_iso: str, destino=None):
    """Tenta baixar o DOP de uma data específica (YYYY-MM-DD).

    Procura a data em DADOS_ULTIMA_DATA verificando campos comuns (data,
    data_publicacao, data_edicao). Se não achar, levanta erro com instrução
    pro usuário inspecionar o portal.
    """
    destino = destino or DESTINO
    with requests.Session() as s:
        itens = _dop_pegar_itens_home(s)
        for item in itens:
            for key in ("data", "data_publicacao", "data_edicao"):
                if str(item.get(key, "")).startswith(data_iso) and item.get("suplemento") == "":
                    nome = f"DOP_{data_iso.replace('-', '_')}.pdf"
                    return _dop_baixar_pdf(s, item["id"], nome, destino)
        raise RuntimeError(
            f"DOP {data_iso} não está exposto em DADOS_ULTIMA_DATA. "
            "O portal só expõe edições recentes pela rota pública. "
            f"Pra arquivo histórico: abra {DOP_BASE} no navegador, navegue até a edição "
            "desejada, pegue o ID em DevTools → Network e use "
            "_dop_baixar_pdf(session, id, nome, destino) direto."
        )

def baixar_dop_por_edicao(numero: int, destino=None):
    """Tenta baixar o DOP pelo número da edição (procura na home)."""
    destino = destino or DESTINO
    with requests.Session() as s:
        itens = _dop_pegar_itens_home(s)
        for item in itens:
            for key in ("numero", "edicao", "numero_edicao"):
                if str(item.get(key, "")) == str(numero) and item.get("suplemento") == "":
                    nome = f"DOP_edicao_{numero}.pdf"
                    return _dop_baixar_pdf(s, item["id"], nome, destino)
        raise RuntimeError(
            f"DOP edição {numero} não está exposta em DADOS_ULTIMA_DATA. "
            "Mesma limitação de baixar_dop_por_data: o portal só lista edições recentes. "
            f"Pra arquivo histórico, pegue o ID manualmente em {DOP_BASE} (DevTools → Network) "
            "e chame _dop_baixar_pdf direto."
        )

def baixar_dop_range(inicio, fim, tipo: str, destino=None):
    """Baixa um intervalo de DOPs.

    tipo='data':   inicio e fim no formato 'YYYY-MM-DD' (inclusivos).
    tipo='edicao': inicio e fim como inteiros (inclusivos).

    Falhas individuais são logadas mas não interrompem o range.
    """
    destino = destino or DESTINO
    saidas = []
    if tipo == "data":
        d_ini = datetime.date.fromisoformat(str(inicio))
        d_fim = datetime.date.fromisoformat(str(fim))
        d = d_ini
        while d <= d_fim:
            try:
                saidas.append(baixar_dop_por_data(d.isoformat(), destino))
            except Exception as e:
                print(f"[DOP] aviso ({d}): {e}")
            d += datetime.timedelta(days=1)
    elif tipo == "edicao":
        for n in range(int(inicio), int(fim) + 1):
            try:
                saidas.append(baixar_dop_por_edicao(n, destino))
            except Exception as e:
                print(f"[DOP] aviso (edicao {n}): {e}")
    else:
        raise ValueError(
            f"baixar_dop_range: tipo={tipo!r} inválido — use 'data' (YYYY-MM-DD) "
            "ou 'edicao' (int)."
        )
    return saidas


## 4. DCM — Diário da Câmara Municipal

Fonte: <https://dcmdigital.camara.rj.gov.br>

### Por que Selenium e não requests?

A home do DCM é uma SPA renderizada por JavaScript. Um `curl` direto pega só o shell HTML sem os dados — precisa de browser real esperando o JS executar.

### Como busca por número funciona

1. Abre a home, espera o `<select id="yDcm2">` ficar visível.
2. Seleciona o ano e preenche `<input id="inputEdicao">` com o número.
3. Clica `<button id="btnEd">` e espera o `<div id="corpoModal">`.
4. Cada `<figure>` no modal tem um `<a href="/download/{id}">` — coleta todos.
5. Baixa cada caderno via `requests` e, se houver mais de um, une com `PyPDF2.PdfMerger`.

### Como busca por data funciona

O site não tem busca por data — só por número. `baixar_dcm_por_data` é **best-effort**:

1. Lê a edição atual da home (âncora: edição N = data D).
2. Estima o número-alvo contando dias úteis entre D e a data pedida.
3. Baixa o palpite, lê a data na 1ª página do PDF, ajusta e tenta de novo (até 5x).
4. Se feriado/recesso desalinhar além disso, levanta erro pedindo pra usar `baixar_dcm_edicao(ano, numero)` direto com o número descoberto manualmente.

### Retry — por que existe

O CDN do DCM cai em **`SSL: UNEXPECTED_EOF_WHILE_READING`** no meio do download de vez em quando. `DCM_TENTATIVAS=3` com `DCM_BACKOFF_S=6` segundos resolve sem intervenção.

### Funções públicas

| Função | Modo |
|---|---|
| `baixar_dcm_hoje()` | `MODO="hoje"` |
| `baixar_dcm_edicao(ano, numero)` | `MODO="edicao"` (por número) |
| `baixar_dcm_por_data("YYYY-MM-DD")` | `MODO="edicao"` (por data — best-effort) |
| `baixar_dcm_range(ano, inicio, fim, tipo="edicao")` | `MODO="range"` por número |
| `baixar_dcm_range(ano, "YYYY-MM-DD", "YYYY-MM-DD", tipo="data")` | `MODO="range"` por data |


In [ ]:
DCM_BASE = "https://dcmdigital.camara.rj.gov.br/"
DCM_UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
          "(KHTML, like Gecko) Chrome/120.0 Safari/537.36")
DCM_REQ_HEADERS = {"User-Agent": DCM_UA, "Referer": DCM_BASE,
                   "Accept": "application/pdf,*/*"}

# Retry config: o CDN do DCM as vezes devolve SSL EOF no meio do download.
DCM_TENTATIVAS = 3
DCM_BACKOFF_S = 6

def _dcm_novo_driver():
    """Cria um WebDriver Chrome novo em modo headless.
    O Selenium 4.6+ resolve o chromedriver automaticamente via selenium-manager."""
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options as ChromeOptions
    opts = ChromeOptions()
    for arg in ("--headless", "--no-sandbox", "--disable-dev-shm-usage",
                "--disable-gpu", "--window-size=1920,1080"):
        opts.add_argument(arg)
    opts.add_argument(f"--user-agent={DCM_UA}")
    return webdriver.Chrome(options=opts)

def _dcm_retry(fn, *args, label="", **kwargs):
    """Executa fn com até DCM_TENTATIVAS tentativas, com backoff fixo.
    Selenium + DCM são intermitentemente flaky (race entre drivers, SSL EOF do CDN)."""
    ultimo_erro = None
    for tentativa in range(1, DCM_TENTATIVAS + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            ultimo_erro = e
            print(f"[DCM] {label} tentativa {tentativa}/{DCM_TENTATIVAS} falhou: "
                  f"{type(e).__name__}: {str(e)[:120]}")
            if tentativa < DCM_TENTATIVAS:
                time.sleep(DCM_BACKOFF_S)
    raise ultimo_erro

def _dcm_baixar_urls(urls, prefixo, destino):
    """Baixa cada URL como caderno parcial e une em um único PDF final.

    Se vier 1 caderno: renomeia direto pro nome final.
    Se vierem N: unifica com PdfMerger e remove os temporários."""
    parciais = []
    with requests.Session() as s:
        s.headers.update(DCM_REQ_HEADERS)
        for i, url in enumerate(urls, 1):
            try:
                r = s.get(url, timeout=120, verify=False)
                r.raise_for_status()
            except requests.exceptions.RequestException as e:
                print(f"[DCM] erro baixando {url}: {e}")
                continue
            if "application/pdf" not in r.headers.get("Content-Type", "").lower():
                print(f"[DCM] aviso: {url} não retornou PDF, pulando.")
                continue
            tmp = os.path.join(destino, f"_tmp_{prefixo}_{i}.pdf")
            with open(tmp, "wb") as f:
                f.write(r.content)
            parciais.append(tmp)
    if not parciais:
        raise RuntimeError(
            f"Nenhum caderno do DCM baixado com sucesso (prefixo={prefixo}). "
            "Verifique conectividade e se o CDN do DCM está respondendo. "
            "Se persistir, aumente DCM_TENTATIVAS na celula de configuracao do DCM."
        )
    if len(parciais) == 1:
        final = os.path.join(destino, f"{prefixo}.pdf")
        shutil.move(parciais[0], final)
    else:
        final = os.path.join(destino, f"{prefixo}_unificado.pdf")
        merger = PdfMerger()
        try:
            for p in parciais:
                merger.append(p)
            merger.write(final)
        finally:
            merger.close()
        for p in parciais:
            try: os.remove(p)
            except OSError: pass
    print(f"[DCM] OK {final}")
    return final

def _dcm_pegar_estado_atual():
    """Lê a home do DCM e retorna (numero_edicao_atual: int, data: datetime.date).
    Sem baixar PDFs — usado como âncora pela busca por data."""
    driver = _dcm_novo_driver()
    try:
        driver.get(DCM_BASE)
        time.sleep(5)
        html = driver.page_source
    finally:
        driver.quit()
    m = re.search(
        r'<span id="edAtual">\s*Última Edição:\s*(\d+)\s*</span>\s*<span>\s*-\s*(\d{2}/\d{2}/\d{4})',
        html
    )
    if not m:
        raise RuntimeError(
            f"Não consegui extrair número/data da edição atual do DCM em {DCM_BASE}. "
            "O regex em _dcm_pegar_estado_atual depende do <span id=\"edAtual\">; "
            "abra a home no devtools e confira se esse span ainda existe."
        )
    numero = int(m.group(1))
    data = datetime.datetime.strptime(m.group(2), "%d/%m/%Y").date()
    return numero, data, html

def _baixar_dcm_hoje_uma_vez(destino):
    """Implementação sem retry — usada via _dcm_retry."""
    edicao, data, html = _dcm_pegar_estado_atual()
    data_str = data.strftime("%d_%m_%Y")
    print(f"[DCM] última edição: {edicao} ({data_str})")

    # dois layouts possíveis na home: caderno único (comentário HTML marcador)
    # ou múltiplos cadernos (botões com attr arg="...")
    m_unico = re.search(
        r'<!-- Só tem 01 Caderno -->.*?<a href="(https://dcmdigital\.camara\.rj\.gov\.br/download/[^"]+)"',
        html, re.DOTALL
    )
    if m_unico:
        urls = [m_unico.group(1).strip()]
    else:
        ids = re.findall(
            r'class="btn btn-sm btn-primary btn-busca buscaDcmDoDia".+?arg="(\w+)"',
            html, re.DOTALL
        )
        if not ids:
            raise RuntimeError(
            f"Não encontrei links de caderno na home do DCM ({DCM_BASE}). "
            "Layout pode ter mudado. Inspecione a home com devtools e ajuste os regex em "
            "_baixar_dcm_hoje_uma_vez."
        )
        urls = [urljoin(DCM_BASE, f"download/{a}") for a in ids]

    return _dcm_baixar_urls(urls, f"DCM_{data_str}_ed{edicao}", destino)

def baixar_dcm_hoje(destino=None):
    """Baixa a última edição publicada do DCM (todos os cadernos do dia, unificados)."""
    destino = destino or DESTINO
    return _dcm_retry(_baixar_dcm_hoje_uma_vez, destino, label="hoje")

def _baixar_dcm_edicao_uma_vez(ano, numero, destino):
    """Implementação sem retry — usada via _dcm_retry."""
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import Select, WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.common.exceptions import TimeoutException

    driver = _dcm_novo_driver()
    urls = []
    try:
        driver.get(DCM_BASE)
        wait = WebDriverWait(driver, 40)
        # 1) ano
        wait.until(EC.visibility_of_element_located((By.ID, "yDcm2")))
        time.sleep(2)
        Select(driver.find_element(By.ID, "yDcm2")).select_by_value(str(ano))
        time.sleep(1)
        # 2) numero
        inp = wait.until(EC.visibility_of_element_located((By.ID, "inputEdicao")))
        inp.clear()
        inp.send_keys(str(numero))
        # 3) clica OK e espera o modal
        wait.until(EC.element_to_be_clickable((By.ID, "btnEd"))).click()
        wait.until(EC.visibility_of_element_located((By.ID, "corpoModal")))
        time.sleep(2)
        # 4) coleta links de download (cada figure = 1 caderno)
        try:
            figs = wait.until(EC.presence_of_all_elements_located(
                (By.XPATH, "//div[@id='corpoModal']/figure")))
        except TimeoutException:
            figs = []
        for fig in figs:
            try:
                link = fig.find_element(By.XPATH, ".//a[starts-with(@href, '/download/')]")
                href = link.get_attribute("href")
                if href:
                    urls.append(urljoin(DCM_BASE, href))
            except Exception:
                continue
    finally:
        driver.quit()

    if not urls:
        raise RuntimeError(
            f"DCM edição {numero}/{ano}: nenhum caderno encontrado. "
            "Confira se essa edição existe — alguns anos têm furos no início ou fim. "
            f"Veja a lista oficial em {DCM_BASE}."
        )
    return _dcm_baixar_urls(urls, f"DCM_ed{numero}_ano{ano}", destino)

def baixar_dcm_edicao(ano: int, numero: int, destino=None):
    """Baixa o DCM de uma edição específica (ano + número). Faz retry automático
    em flakiness do Selenium ou SSL EOF do CDN."""
    destino = destino or DESTINO
    return _dcm_retry(_baixar_dcm_edicao_uma_vez, ano, numero, destino,
                      label=f"ed {numero}/{ano}")

def _dias_uteis_entre(d1: datetime.date, d2: datetime.date) -> int:
    """Dias úteis (seg-sex) de d1 a d2. Positivo se d2 > d1. Ignora feriados —
    se a heurística furar, baixar_dcm_por_data ajusta na próxima tentativa."""
    if d1 == d2:
        return 0
    sinal = 1 if d2 > d1 else -1
    inicio, fim = (d1, d2) if sinal > 0 else (d2, d1)
    dias = 0
    cur = inicio + datetime.timedelta(days=1)
    while cur <= fim:
        if cur.weekday() < 5:
            dias += 1
        cur += datetime.timedelta(days=1)
    return sinal * dias

def _dcm_extrair_data_do_pdf(caminho_pdf):
    """Extrai a primeira data DD/MM/AAAA das 2 primeiras páginas do PDF.
    Retorna datetime.date ou None se não achar / falhar."""
    try:
        reader = PdfReader(caminho_pdf)
        texto = ""
        for pagina in reader.pages[:2]:
            texto += (pagina.extract_text() or "") + "\n"
        m = re.search(r"\b(\d{2})/(\d{2})/(\d{4})\b", texto)
        if not m:
            return None
        return datetime.date(int(m.group(3)), int(m.group(2)), int(m.group(1)))
    except Exception:
        return None

def baixar_dcm_por_data(data_iso: str, destino=None):
    """Baixa o DCM publicado em uma data específica (YYYY-MM-DD).

    Best-effort — o site do DCM só aceita busca por número, não por data:

    1. Lê edição atual da home (âncora número↔data).
    2. Estima a edição-alvo contando dias úteis entre hoje e a data pedida.
    3. Baixa o palpite, lê a data da primeira página do PDF, e ajusta até
       no máximo 5 tentativas.

    Se feriados/recessos confundirem além disso, levanta erro pedindo pra
    usar baixar_dcm_edicao(ano, numero) com o número correto descoberto
    manualmente no portal.
    """
    destino = destino or DESTINO
    alvo = datetime.date.fromisoformat(data_iso)
    numero_hoje, data_hoje, _ = _dcm_pegar_estado_atual()
    print(f"[DCM] âncora: edição {numero_hoje} = {data_hoje.isoformat()}; alvo = {alvo.isoformat()}")
    if alvo > data_hoje:
        raise RuntimeError(
            f"Data alvo {alvo} é posterior à última edição publicada ({data_hoje}). "
            "Aguarde a publicação ou use baixar_dcm_hoje()."
        )
    palpite = numero_hoje + _dias_uteis_entre(data_hoje, alvo)
    if palpite < 1:
        raise RuntimeError(f"Heurística produziu edição inválida ({palpite}) para {alvo}.")

    tentados = set()
    for tentativa in range(1, 6):
        while palpite in tentados:
            palpite += 1
        tentados.add(palpite)
        print(f"[DCM] por_data tentativa {tentativa}/5 — testando edição {palpite}")
        pdf = baixar_dcm_edicao(alvo.year, palpite, destino)
        data_pdf = _dcm_extrair_data_do_pdf(pdf)
        if data_pdf == alvo:
            final = os.path.join(destino, f"DCM_data_{alvo.strftime('%Y_%m_%d')}_ed{palpite}.pdf")
            if pdf != final:
                shutil.move(pdf, final)
            print(f"[DCM] OK {final} (data confirmada no conteúdo)")
            return final
        if data_pdf is None:
            final = os.path.join(destino, f"DCM_data_{alvo.strftime('%Y_%m_%d')}_ed{palpite}_palpite.pdf")
            if pdf != final:
                shutil.move(pdf, final)
            print(f"[DCM] aviso: não consegui ler data do PDF — entregando como palpite: {final}")
            return final
        diff = _dias_uteis_entre(data_pdf, alvo)
        print(f"[DCM] PDF tem data {data_pdf}, diferença em dias úteis = {diff:+d}; ajustando")
        try: os.remove(pdf)
        except OSError: pass
        if diff == 0:
            palpite += 1 if alvo > data_pdf else -1
        else:
            palpite += diff
        if palpite < 1:
            break
    raise RuntimeError(
        f"Não consegui localizar a edição do DCM de {alvo} em 5 tentativas. "
        "Feriados ou recessos podem ter desalinhado a heurística. "
        f"Acesse {DCM_BASE} para descobrir o número da edição "
        f"e use baixar_dcm_edicao({alvo.year}, NUMERO)."
    )

def baixar_dcm_range(ano: int, inicio, fim, tipo: str = "edicao", destino=None):
    """Baixa um intervalo de DCMs.

    tipo='edicao' (default): inicio/fim são números inteiros no mesmo `ano`.
    tipo='data':             inicio/fim são 'YYYY-MM-DD' (inclusivos); `ano`
                             é ignorado. Pula sábados e domingos. Cada dia
                             vai por baixar_dcm_por_data (heurística + verificação).

    Falhas individuais são logadas mas não interrompem o range.
    """
    destino = destino or DESTINO
    saidas = []
    if tipo == "edicao":
        for n in range(int(inicio), int(fim) + 1):
            try:
                saidas.append(baixar_dcm_edicao(ano, n, destino))
            except Exception as e:
                print(f"[DCM] aviso (edicao {n}): {e}")
    elif tipo == "data":
        d_ini = datetime.date.fromisoformat(str(inicio))
        d_fim = datetime.date.fromisoformat(str(fim))
        d = d_ini
        while d <= d_fim:
            if d.weekday() < 5:
                try:
                    saidas.append(baixar_dcm_por_data(d.isoformat(), destino))
                except Exception as e:
                    print(f"[DCM] aviso ({d}): {e}")
            d += datetime.timedelta(days=1)
    else:
        raise ValueError("tipo deve ser 'edicao' ou 'data'")
    return saidas


## 5. Runner

Esta célula é o dispatcher: lê as variáveis da Seção 1 e chama as funções certas. Você raramente precisa mexer aqui.

Falha de uma fonte (DOP ou DCM) não derruba a outra — cada bloco está em seu próprio `try/except`. Útil quando, por exemplo, o DCM está com instabilidade mas você quer pelo menos o DOP do dia.


In [ ]:
_MODOS_VALIDOS  = {"hoje", "edicao", "range"}
_FONTES_VALIDAS = {"dcm", "dop", "ambos"}
_RANGE_TIPOS    = {"edicao", "data"}

def _validar_config():
    if MODO not in _MODOS_VALIDOS:
        raise ValueError(f"MODO={MODO!r} inválido — use um de {sorted(_MODOS_VALIDOS)}.")
    if FONTE not in _FONTES_VALIDAS:
        raise ValueError(f"FONTE={FONTE!r} inválida — use uma de {sorted(_FONTES_VALIDAS)}.")
    if MODO == "edicao" and not EDICAO_DATA and not EDICAO_NUMERO:
        raise ValueError("MODO='edicao' precisa de EDICAO_DATA ou EDICAO_NUMERO.")
    if MODO == "range":
        if RANGE_TIPO not in _RANGE_TIPOS:
            raise ValueError(f"RANGE_TIPO={RANGE_TIPO!r} inválido — use 'edicao' ou 'data'.")
        if RANGE_TIPO == "data":
            try:
                datetime.date.fromisoformat(str(RANGE_INICIO))
                datetime.date.fromisoformat(str(RANGE_FIM))
            except ValueError:
                raise ValueError(
                    f"RANGE_TIPO='data' exige RANGE_INICIO/RANGE_FIM no formato 'YYYY-MM-DD' "
                    f"(recebi {RANGE_INICIO!r}, {RANGE_FIM!r})."
                )

def rodar():
    _validar_config()
    saidas = []

    # ---------- DOP ----------
    if FONTE in ("dop", "ambos"):
        try:
            if MODO == "hoje":
                saidas.append(baixar_dop_hoje(DESTINO))
            elif MODO == "edicao":
                # data tem precedência se ambas estiverem setadas
                if EDICAO_DATA:
                    saidas.append(baixar_dop_por_data(EDICAO_DATA, DESTINO))
                else:
                    saidas.append(baixar_dop_por_edicao(EDICAO_NUMERO, DESTINO))
            elif MODO == "range":
                saidas += baixar_dop_range(RANGE_INICIO, RANGE_FIM, RANGE_TIPO, DESTINO)
        except Exception as e:
            print(f"[DOP] falhou: {e}")

    # ---------- DCM ----------
    if FONTE in ("dcm", "ambos"):
        try:
            if MODO == "hoje":
                saidas.append(baixar_dcm_hoje(DESTINO))
            elif MODO == "edicao":
                # data tem precedência (heurística + verificação). Número é a rota direta.
                if EDICAO_DATA:
                    saidas.append(baixar_dcm_por_data(EDICAO_DATA, DESTINO))
                else:
                    saidas.append(baixar_dcm_edicao(EDICAO_ANO, EDICAO_NUMERO, DESTINO))
            elif MODO == "range":
                # tipo='edicao' usa EDICAO_ANO + ints; tipo='data' usa strings ISO
                saidas += baixar_dcm_range(EDICAO_ANO, RANGE_INICIO, RANGE_FIM,
                                           tipo=RANGE_TIPO, destino=DESTINO)
        except Exception as e:
            print(f"[DCM] falhou: {e}")

    return [s for s in saidas if s]

pdfs_baixados = rodar()
print(f"\n{len(pdfs_baixados)} PDF(s) baixado(s).")


## 6. Resultado

Tabela com os arquivos baixados nesta execução. Útil pra confirmar que tudo está coerente antes de subir pra um drive, mandar pra alguém ou alimentar uma próxima etapa de pipeline (extração, resumo, classificação).


In [ ]:
import pandas as pd

linhas = []
for p in pdfs_baixados:
    try:
        n_paginas = len(PdfReader(p).pages)
    except Exception:
        n_paginas = "?"
    linhas.append({
        "arquivo":   os.path.basename(p),
        "tamanho_MB": round(os.path.getsize(p) / 1024 / 1024, 2),
        "paginas":   n_paginas,
        "modificado": datetime.datetime.fromtimestamp(
            os.path.getmtime(p)).strftime("%Y-%m-%d %H:%M"),
    })

if linhas:
    df = pd.DataFrame(linhas)
    try:
        from IPython.display import display
        display(df)
    except ImportError:
        print(df.to_string(index=False))
else:
    print("Nenhum PDF foi baixado nesta execução.")


## 7. Troubleshooting

### `ImportError: Falta o pacote ...`
Rodou "Run All" sem instalar as dependências. Volte na Seção 2, descomente as duas linhas (`pip install …` e `apt-get install …`) e rode essa célula primeiro.

### `ValueError: MODO=... inválido` / `FONTE=... inválida`
O runner valida antes de tocar a rede. Aceitos: `MODO` ∈ {`hoje`, `edicao`, `range`}; `FONTE` ∈ {`dcm`, `dop`, `ambos`}; `RANGE_TIPO` ∈ {`edicao`, `data`}.

### `SessionNotCreatedException: no chrome binary at /usr/bin/google-chrome`
Selenium não encontrou o Chrome.
- **Colab / Linux:** `apt install google-chrome-stable` ou aponte explicitamente: `opts.binary_location = "/usr/bin/chromium"`.
- **macOS:** o Chrome em `/Applications/Google Chrome.app` costuma ser detectado automaticamente. Se não, aponte com `opts.binary_location = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"`.
- **GitHub Actions:** use `browser-actions/setup-chrome@v1` em vez do apt — o `chromium-browser` do Ubuntu é stub de snap e não funciona em CI.

### `Variável DADOS_ULTIMA_DATA não encontrada na home do DOP`
O portal mudou a estrutura. Abra `https://doweb.rio.rj.gov.br` no devtools, procure `DADOS_ULTIMA_DATA` no `<script>` e ajuste o regex em `_dop_pegar_itens_home`.

### `DOP <data> não está exposto em DADOS_ULTIMA_DATA`
Limitação do portal — só expõe edições recentes. Pra arquivo histórico, pegue o ID no devtools (Network) e chame `_dop_baixar_pdf(session, id, nome, destino)` direto.

### `SSL: UNEXPECTED_EOF_WHILE_READING` no DCM
Flakiness do CDN. O retry interno (`DCM_TENTATIVAS=3`) já trata. Se persistir, aumente: `DCM_TENTATIVAS = 5; DCM_BACKOFF_S = 10`.

### `TimeoutException` no DCM ao buscar `yDcm2`/`inputEdicao`/`corpoModal`
A página não terminou de renderizar a tempo. Aumente o `WebDriverWait(driver, 40)` ou o `time.sleep(5)` na home.

### `Não consegui localizar a edição do DCM de <data> em 5 tentativas`
A heurística de `baixar_dcm_por_data` se baseia em "1 edição por dia útil". Feriados, recessos ou edições extraordinárias desalinham. Abra `https://dcmdigital.camara.rj.gov.br`, descubra o número da edição daquela data e chame `baixar_dcm_edicao(ano, numero)` direto.

### DCM salvou com sufixo `_palpite`
A heurística baixou o PDF mas não conseguiu ler a data na 1ª página pra confirmar (texto não-extraível ou layout incomum). Abra e verifique: se for a edição certa, renomeie removendo `_palpite`; se não, siga o caminho do erro acima.

### `Data alvo X é posterior à última edição publicada (Y)`
Você pediu uma data que ainda não saiu — DCM publica em dia útil. Use `baixar_dcm_hoje()` ou espere a publicação.

### O PDF salvou mas está vazio / corrompido
O `Content-Type` é validado nas duas pontas. Se ainda assim acontecer, abra o arquivo e veja se começa com `%PDF-`. Em geral indica que o servidor devolveu HTML de erro com `Content-Type: application/pdf` (raro).

---

## 8. Como adaptar pra outros diários oficiais

O padrão é genérico:

1. **Achar o endpoint de download.** Use o devtools do navegador na aba *Network* enquanto baixa manualmente uma edição.
2. **Descobrir como o portal lista edições.** Variável JS na home? API JSON? Formulário HTML?
3. **Escolher entre HTTP puro ou Selenium.** Se a listagem está no HTML renderizado pelo servidor, `requests` resolve. Se for SPA, Selenium.
4. **Padronizar o nome do arquivo.** Prefira `{ORIGEM}_{data}_{edicao}.pdf` pra ordenar fácil.
5. **Tratar PDFs múltiplos.** Diários costumam ter "cadernos" — junte com `PyPDF2.PdfMerger` ou salve separados, conforme sua necessidade downstream.
6. **Adicionar retry.** Sites de governo são notoriamente instáveis em horário de pico.
7. **Se o portal não expõe busca por data**, ancore na edição mais recente e palpite por dias úteis (igual `baixar_dcm_por_data`). Confirme abrindo o PDF — não confie só na heurística.
